In [ ]:
"""
prototype_ensemble_evaluation.py
Quick evaluation using existing models to prototype the final pipeline.
"""
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, fbeta_score

# Paths
MODELS_DIR = Path("../models")
SPLITS_DIR = Path("../data/splits")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_existing_models(mode, text_amount="medium"):
    """Load your existing models (even with leakage) for prototyping."""
    
    # MLP
    mlp_path = MODELS_DIR / f"best_mlp_{mode}_{text_amount}.pth"
    mlp_state_dict = torch.load(mlp_path, map_location=device)
    
    # CNN
    cnn_path = MODELS_DIR / f"best_cnn_densenet_{mode}.pth"
    cnn_state_dict = torch.load(cnn_path, map_location=device)
    
    return mlp_state_dict, cnn_state_dict


def quick_ensemble_evaluation(mode, text_amount="medium"):
    """
    Prototype ensemble evaluation.
    Uses whatever test data is available to show the pipeline.
    """
    print(f"\n{'='*70}")
    print(f"PROTOTYPE ENSEMBLE EVALUATION - {mode.upper()}")
    print(f"{'='*70}")
    
    # 1. Load models
    print("\n📦 Loading models...")
    mlp_state_dict, cnn_state_dict = load_existing_models(mode, text_amount)
    
    # Recreate models
    from src.models.mlp import MLP
    from src.models.cnn import FinetunedDenseNet
    
    # MLP - determine input dim from state dict
    input_dim = mlp_state_dict['network.0.weight'].shape[1]
    mlp_model = MLP(input_nodes=input_dim).to(device)
    mlp_model.load_state_dict(mlp_state_dict)
    mlp_model.eval()
    
    # CNN
    cnn_model = FinetunedDenseNet().to(device)
    cnn_model.load_state_dict(cnn_state_dict)
    cnn_model.eval()
    
    print(f"  ✅ MLP input dim: {input_dim}")
    print(f"  ✅ CNN loaded")
    
    # 2. Load test data (use validation if test doesn't exist)
    print("\n📊 Loading test data...")
    try:
        test_df = pd.read_csv(SPLITS_DIR / f"{mode}_test.csv")
        print(f"  ✅ Using held-out test set: {len(test_df)} samples")
    except:
        # Fallback to validation set for prototyping
        test_df = pd.read_csv(SPLITS_DIR / f"{mode}_val.csv")
        print(f"  ⚠️ Using validation set (test not found): {len(test_df)} samples")
    
    print(f"     Positives: {test_df['label'].sum()} ({test_df['label'].mean():.1%})")
    
    # 3. Get MLP predictions
    print("\n🔮 Getting MLP predictions...")
    from src.data.nlp_preprocessing import build_nlp_pipeline, prepare_text_features
    
    nlp = build_nlp_pipeline()
    test_df = prepare_text_features(test_df.copy(), nlp, mode=text_amount)
    
    # Load vectorizer
    vectorizer_path = MODELS_DIR / f"vectorizer_{mode}_{text_amount}.pkl"
    with open(vectorizer_path, 'rb') as f:
        vectorizer = pickle.load(f)
    
    X_test = vectorizer.transform(test_df["text_clean"]).toarray()
    y_test = test_df["label"].values
    
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        mlp_logits = mlp_model(X_tensor).squeeze()
        mlp_probs = torch.sigmoid(mlp_logits).cpu().numpy()
    
    print(f"  ✅ MLP predictions: {len(mlp_probs)} samples")
    
    # 4. Get CNN predictions
    print("\n🖼️ Getting CNN predictions...")
    from src.data.dataset import ChestXRayDataset
    from src.data.loader import get_transforms_grayscale
    from torch.utils.data import DataLoader
    
    _, test_transforms = get_transforms_grayscale()
    test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    cnn_probs = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            logits = cnn_model(images).squeeze()
            probs = torch.sigmoid(logits).cpu().numpy()
            cnn_probs.extend(probs if hasattr(probs, '__len__') else [probs])
    
    cnn_probs = np.array(cnn_probs)
    print(f"  ✅ CNN predictions: {len(cnn_probs)} samples")
    
    # 5. Ensemble
    print("\n🎯 Ensemble Results:")
    print("-" * 50)
    
    # Simple average
    ensemble_probs = 0.5 * mlp_probs + 0.5 * cnn_probs
    
    # Weighted (tune these based on validation performance)
    ensemble_weighted = 0.6 * mlp_probs + 0.4 * cnn_probs  # Text often better
    
    # Calculate metrics
    mlp_auc = roc_auc_score(y_test, mlp_probs)
    cnn_auc = roc_auc_score(y_test, cnn_probs)
    ensemble_auc = roc_auc_score(y_test, ensemble_probs)
    weighted_auc = roc_auc_score(y_test, ensemble_weighted)
    
    print(f"  MLP AUC:           {mlp_auc:.4f}")
    print(f"  CNN AUC:           {cnn_auc:.4f}")
    print(f"  Ensemble (0.5/0.5): {ensemble_auc:.4f} ({ensemble_auc - max(mlp_auc, cnn_auc):+.4f} vs best single)")
    print(f"  Ensemble (0.6/0.4): {weighted_auc:.4f} ({weighted_auc - max(mlp_auc, cnn_auc):+.4f} vs best single)")
    
    # 6. Threshold analysis
    print("\n📈 Threshold Analysis (Ensemble 0.5/0.5):")
    print("-" * 50)
    
    for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
        preds = (ensemble_probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        acc = (tp + tn) / len(y_test)
        f2 = fbeta_score(y_test, preds, beta=2)  # Weight recall higher
        
        print(f"  t={t:.2f} | Acc: {acc:.1%} | Sens: {sens:.1%} | Spec: {spec:.1%} | F2: {f2:.3f} | FN: {fn:3d} | FP: {fp:3d}")
    
    # 7. Best threshold for triage (minimize false negatives)
    print("\n🚑 Triage-Optimized Threshold (minimize FN):")
    best_t = 0.1  # Start low
    best_fn = float('inf')
    for t in np.arange(0.05, 0.51, 0.01):
        preds = (ensemble_probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
        if fn < best_fn:
            best_fn = fn
            best_t = t
    
    preds_best = (ensemble_probs >= best_t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds_best).ravel()
    print(f"  Optimal threshold: {best_t:.2f}")
    print(f"  Sensitivity: {tp/(tp+fn):.1%} | Specificity: {tn/(tn+fp):.1%}")
    print(f"  False Negatives: {fn} | False Positives: {fp}")
    
    return {
        'mlp_auc': mlp_auc,
        'cnn_auc': cnn_auc,
        'ensemble_auc': ensemble_auc,
        'weighted_auc': weighted_auc,
        'mlp_probs': mlp_probs,
        'cnn_probs': cnn_probs,
        'ensemble_probs': ensemble_probs,
        'y_test': y_test
    }


def plot_ensemble_roc(results, mode):
    """Plot ROC curves for all three models."""
    import matplotlib.pyplot as plt
    from sklearn.metrics import RocCurveDisplay
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    RocCurveDisplay.from_predictions(
        results['y_test'], results['mlp_probs'], 
        name=f"MLP (AUC={results['mlp_auc']:.3f})", ax=ax
    )
    RocCurveDisplay.from_predictions(
        results['y_test'], results['cnn_probs'], 
        name=f"CNN (AUC={results['cnn_auc']:.3f})", ax=ax
    )
    RocCurveDisplay.from_predictions(
        results['y_test'], results['ensemble_probs'], 
        name=f"Ensemble (AUC={results['ensemble_auc']:.3f})", 
        linewidth=3, color='red', ax=ax
    )
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves - {mode.upper()} Mode")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"ensemble_roc_{mode}.png", dpi=150)
    plt.show()


# Run the prototype evaluation
if __name__ == "__main__":
    image_dir = Path("../IU-xray/images")  # Update this path
    
    results = {}
    for mode in ["abnormal", "pneumonia"]:
        results[mode] = quick_ensemble_evaluation(mode, text_amount="medium")
        plot_ensemble_roc(results[mode], mode)
    
    # Comparison table
    print("\n" + "="*70)
    print("FINAL COMPARISON TABLE")
    print("="*70)
    print(f"{'Mode':<12} {'MLP AUC':<10} {'CNN AUC':<10} {'Ensemble AUC':<12} {'Improvement':<12}")
    print("-"*70)
    for mode in ["abnormal", "pneumonia"]:
        r = results[mode]
        improvement = r['ensemble_auc'] - max(r['mlp_auc'], r['cnn_auc'])
        print(f"{mode:<12} {r['mlp_auc']:.4f}     {r['cnn_auc']:.4f}     {r['ensemble_auc']:.4f}        {improvement:+.4f}")